# 확장 시각화 — 텍스트 워드클라우드(WordCloud) · 지도 시각화(Folium)


**직접 풀어보는 12문제:** 정답 코드는 넣지 않았습니다. 준비 코드만 제공하며 각 문제의 빈 셀에 풀이를 작성합니다. 점검 숫자는 아래 가상 데이터에 대한 값입니다. 이 노트북의 준비 데이터에만 점검 숫자를 적용하세요.

각 노트북은 독립적으로 사용할 수 있습니다. 위에서 아래로 진행하고, 실험에는 복사본을 사용하세요. 개별 문제에서 만든 결과는 다른 이름에 저장해 원본 변수를 덮어쓰지 않는 것을 권합니다.

## 처음 배우는 분을 위한 안내

토큰은 문장을 쪼갠 단어 단위, 불용어는 분석에서 빼기로 정한 단어, Counter는 출현 횟수를 세는 도구입니다. 여기서는 공백 분리만 사용하며 형태소 분석은 필요하지 않습니다. 위도는 남북 위치, 경도는 동서 위치이며 지도 좌표는 [위도, 경도] 순서입니다.

**학습 순서:** 문제 1 ~ 4는 필수 기초, 5 ~ 12는 선택 도전입니다. 5 ~ 6번은 한글 글꼴이, 7번 이후 지도 화면은 인터넷이 필요할 수 있습니다. 인터넷이 없어도 마커 데이터와 HTML 생성 여부는 확인할 수 있습니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

**지도 화면 주의:** 배경 지도와 Leaflet 스크립트는 외부 서버를 사용합니다. 네트워크가 막혀 빈 배경이 나와도 마커 개수·좌표와 HTML 저장 성공을 따로 확인하세요. 글꼴 파일이 없으면 5 ~ 6번 워드클라우드는 잠시 건너뛰고 1 ~ 4번 표와 7번 이후 지도를 진행할 수 있습니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import folium
from folium.plugins import MarkerCluster

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
plt.rcParams['axes.unicode_minus'] = False

# 지정 글꼴이 없는 환경에서는 설치된 한글 글꼴 파일을 FONT_PATH에 지정합니다.
from pathlib import Path
if not Path(FONT_PATH).is_file():
    candidates = [p for pattern in ('/System/Library/Fonts/*Apple*','/usr/share/fonts/truetype/nanum/*.ttf') for p in __import__('glob').glob(pattern) if Path(p).is_file()]
    if candidates:
        FONT_PATH = candidates[0]
    else:
        raise FileNotFoundError('한글 글꼴 파일이 필요합니다. 글꼴 파일을 올린 뒤 FONT_PATH를 해당 경로로 설정하세요.')

# 1. 워드클라우드 실습용 고객 리뷰·문장 리스트
review_lines = [
    "배송 빠름 포장 깔끔 가격 만족 합니다.",
    "배송 빠름 품질 최고 재구매 의사 있습니다,",
    "가격 합리적 배송 빠름 상담 친절 합니다.",
    "포장 깔끔 품질 만족 배송 최고 입니다.",
    "품질 최고 가격 만족 포장 안전 합니다."
]

# 2. Folium 지도 실습용 매장 위치 데이터 (국가 및 위경도 포함, 8행)
stores_df = pd.DataFrame({
    'Store.Name': [
        '시청역점', '광화문점', '을지로입구점', '명동역점',
        '정동점', '강남역점', '부산서면점', '도쿄신주쿠점'
    ],
    'Country': ['KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'KR', 'JP'],
    'Latitude': [37.5658, 37.5715, 37.5660, 37.5609, 37.5672, 37.4979, 35.1578, 35.6938],
    'Longitude': [126.9772, 126.9765, 126.9827, 126.9858, 126.9721, 127.0276, 129.0592, 139.7034]
})

# 3. 공공시설(의류수거함) 위치 현황 가상 데이터 (결측 1행 포함, 6행)
collection_bins = pd.DataFrame({
    'bin_id': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06'],
    'dong': ['둔산동', '둔산동', '갈마동', '월평동', '갈마동', '둔산동'],
    'latitude': [36.3504, 36.3521, 36.3482, 36.3585, None, 36.3512],
    'longitude': [127.3845, 127.3880, 127.3720, 127.3615, None, 127.3860]
})

## 문제 1. 문장 분리(`split`)와 문장부호 제거(`strip`) [필수 기초]

`review_lines`의 각 문장을 공백 기준으로 나누고(`split()`), 단어 양끝의 마침표(`.`)와 쉼표(`,`)를 제거(`strip('.,')`)하여 순서대로 모은 리스트 `raw_tokens`를 만드세요. 전체 단어 개수와 앞 10개 단어를 출력하세요.

**점검 기준:** 5개 문장(각 7어절)에서 문장부호가 제거된 총 35개의 단어가 `raw_tokens`에 저장됩니다.

In [ ]:
# 문제 1 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 2. 분석에서 제외할 단어 제거 후 핵심 단어 리스트 만들기 [필수 기초]

`raw_tokens`에서 이 예제에서 제외하기로 정한 `'합니다'`, `'있습니다'`, `'입니다'` 세 단어를 제외한 리스트 `clean_tokens`를 만드세요. 제거 전후 단어 수를 비교하세요.

**점검 기준:** 불용어 5개가 제외되어 `clean_tokens`의 총 단어 수는 30개, 고유 단어 수는 14개입니다.

In [ ]:
# 문제 2 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 3. `Counter`로 단어 빈도 계산 및 내림차순 정렬 [필수 기초]

`collections.Counter`를 사용해 `clean_tokens`의 단어별 출현 빈도를 세고, 빈도가 높은 순서(`reverse=True`)로 정렬한 리스트 `wc_sorted`를 출력하세요.

**점검 기준:** 가장 많이 등장한 단어는 `'배송'(4회)`이며, 이어서 `'빠름'`, `'포장'`, `'가격'`, `'만족'`, `'품질'`, `'최고'`가 각 3회씩, `'깔끔'`이 2회, `'재구매'`, `'의사'`, `'합리적'`, `'상담'`, `'친절'`, `'안전'`이 각 1회씩 등장합니다.

In [ ]:
# 문제 3 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 4. 단어 빈도 데이터프레임 변환 및 `{단어: 빈도}` 딕셔너리 생성 [필수 기초]

`wc_sorted`를 컬럼명이 `['word', 'freq']`인 데이터프레임 `wcdf`로 만들고, `'word'`를 인덱스로 설정한 뒤 `.to_dict()`를 사용하여 `{단어: 빈도}` 형태의 딕셔너리 `word_freq`를 생성하세요.

**점검 기준:** `wcdf`는 `14행 × 2열`이며, `word_freq['배송'] == 4`, `word_freq['빠름'] == 3`, `word_freq['깔끔'] == 2`인 딕셔너리가 생성됩니다.

In [ ]:
# 문제 4 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 5. 기본 워드클라우드 생성 및 화면 표시 [선택 도전]

`WordCloud(font_path=FONT_PATH, width=800, height=500).generate_from_frequencies(word_freq)`로 워드클라우드 객체를 생성하고 `plt.imshow()`와 `plt.axis('off')`로 출력하세요.

**점검 기준:** 빈도가 가장 높은 `'배송'`이 가장 크게 배치되고 2회 출현 단어들이 중간 크기로 배치된 워드클라우드 이미지가 생성됩니다.

In [ ]:
# 문제 5 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 6. 워드클라우드 배경색(`background_color`)과 컬러맵(`colormap`) 커스터마이징 [선택 도전]

`background_color='white'`, `colormap='autumn'`, `width=900`, `height=500` 옵션을 적용해 새로운 워드클라우드를 생성하고 제목 `'Word Frequency'`를 붙여 출력하세요.

**점검 기준:** 흰색 배경 위에 붉은/주황 계열(`autumn`) 색상으로 키워드가 선명하게 시각화됩니다.

In [ ]:
# 문제 6 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 7. 서울시청 중심 Folium 기본 지도와 빨간색 마커 생성 [선택 도전]

서울시청 좌표 `city_hall = [37.5662952, 126.9779451]`을 중심으로 `zoom_start=14`인 `folium.Map` 객체 `m1`을 생성하고, `tooltip='서울시청'`, `popup='서울시청 위치'`, `icon=folium.Icon(color='red')`인 마커를 추가하세요.

**점검 기준:** `m1` 지도 객체가 생성되며 중심 좌표 `[37.5662952, 126.9779451]`에 빨간색 핀 마커 1개가 등록됩니다.

In [ ]:
# 문제 7 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 8. 배경 지도 타일(`tiles='Esri.WorldImagery'`) 변경 비교 [선택 도전]

서울시청 좌표에 대해 `tiles='Esri.WorldImagery'`(항공·위성 사진 타일) 옵션을 지정한 지도 `m_sat`을 생성하고 서울시청 마커를 추가한 뒤 일반 지도 타일과의 차이를 적으세요.

**점검 기준:** `Esri.WorldImagery` 타일이 적용되어 도로망 지도 대신 위성/항공 영상이 배경으로 표시됩니다(단, 외부 타일 서비스는 제공 정책에 따라 영구 사용이 보장되지 않을 수 있음).

In [ ]:
# 문제 8 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 9. 위도·경도 사각형 범위(`between`)로 서울 시청 주변 매장 필터링 [선택 도전]

`stores_df`에서 국가가 한국(`Country == 'KR'`)이고 위도가 `37.55 ~ 37.58`(`between(37.55, 37.58)`), 경도가 `126.96 ~ 126.99`(`between(126.96, 126.99)`)인 행만 선택해 `['Store.Name', 'Latitude', 'Longitude']` 열을 가진 `seoul_stores` 복사본을 만드세요.

**점검 기준:** 전체 8개 매장 중 도쿄(`JP`), 부산(`35.15`), 강남역(`37.49`) 매장이 제외되어 시청역점·광화문점·을지로입구점·명동역점·정동점 총 5개 매장이 선택됩니다.

In [ ]:
# 문제 9 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 10. `MarkerCluster`와 `iterrows()`로 매장 마커 일괄 추가하기 [선택 도전]

서울시청 중심 지도 `m`에 `MarkerCluster(name='Starbucks')`를 추가하고, `seoul_stores.iterrows()` 반복문을 사용해 5개 매장의 위치 마커(`tooltip=row['Store.Name']`, `popup=folium.Popup(row['Store.Name'], max_width=250)`)를 클러스터에 추가하세요.

**점검 기준:** 클러스터 객체 안에 5개의 매장 마커가 등록되며, 지도 축소 시 군집 배지로 묶이고 확대 시 개별 매장 마커로 펼쳐집니다.

**차근차근:** 새 folium.Map을 m에 저장합니다. cluster=MarkerCluster(...).add_to(m)으로 묶음을 만들고, for _, row in seoul_stores.iterrows()로 한 행씩 읽습니다. 각 행의 [Latitude, Longitude]로 Marker를 만들어 cluster에 add_to 합니다.

In [ ]:
# 문제 10 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 11. 공공시설(의류수거함) 결측 좌표 제거 및 동별 색상 구분 마커 표시 [선택 도전]

`collection_bins`에서 위도·경도 결측치를 제거한 `valid_bins`를 만들고, 중심 좌표 `[36.351, 127.380]`(`zoom_start=14`) 지도에 `'둔산동'`은 파란색(`'blue'`), 그 외 동은 초록색(`'green'`) 아이콘 마커로 표시하세요.

**점검 기준:** 결측 1행(`B05`)이 제외되어 총 5개 수거함이 지도에 표시되며, 둔산동 3개(`B01, B02, B06`)는 파란색, 갈마동·월평동 2개(`B03, B04`)는 초록색 마커로 구분됩니다.

In [ ]:
# 문제 11 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 12. 완성된 Folium 지도를 HTML 파일로 저장하고 검증하기 [선택 도전]

문제 10에서 만든 지도 `m`을 `m.save('seoul_stores_map.html')`로 저장하고, `pathlib.Path('seoul_stores_map.html').exists()`와 파일 크기(`stat().st_size`)를 출력해 정상 저장 여부를 확인하세요.

**점검 기준:** `seoul_stores_map.html` 파일이 생성되며 파일 크기가 0보다 큰 HTML 문서(Leaflet 스크립트 및 마커 좌표 포함)로 저장됩니다.

In [ ]:
# 문제 12 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 선택 도전: 단어 네 개를 표로 검산하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
extra_words = ['배송','배송','가격','포장']
```

### 진행 순서

1. Counter로 단어 출현 횟수를 셉니다.
2. 빈도 합계와 고유 단어 수를 출력합니다.
3. 가장 자주 등장하는 단어를 적습니다. 이 선택 도전은 폰트·지도·외부 파일 없이 표만으로 완료합니다.

**완료 기준:** 배송 2회, 가격·포장 각각 1회. 빈도 합계 4, 고유 단어 3개.

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?